In [1]:
import pandas as pd
import numpy as np

def prepare_lastfm_data():
    print("1. Đang nạp dữ liệu Last.fm (Dữ liệu thô)...")
    # Trong thực tế, bạn sẽ dùng lệnh này để đọc file tải từ Kaggle:
    # df = pd.read_csv('user_artists.dat', sep='\t') 
    
    # Ở đây mình tạo Mock Data chuẩn cấu trúc Last.fm để bạn chạy thử ngay:
    data = {
        'userID': ['U_001', 'U_001', 'U_001', 'U_002', 'U_002', 'U_003', 'U_003', 'U_004'],
        'itemID': ['T_A', 'T_B', 'T_C', 'T_A', 'T_C', 'T_B', 'T_D', 'T_A'],
        'playcount': [500, 15, 2, 350, 60, 1, 90, 25] # Số lần nghe chênh lệch rất lớn
    }
    df = pd.DataFrame(data)
    print("Dữ liệu thô ban đầu:")
    print(df.head())

    print("\n2. Quy đổi Playcount thành Rating (1 -> 5 sao) bằng Quantile...")
    # Tính các mốc % (20%, 40%, 60%, 80%) của cột playcount để chia đều dữ liệu thành 5 nhóm
    bins = df['playcount'].quantile([0, 0.2, 0.4, 0.6, 0.8, 1.0]).unique()
    
    # Xử lý trường hợp dữ liệu quá ít khiến các mốc bị trùng nhau
    if len(bins) < 6:
        bins = [-1, 5, 20, 50, 100, float('inf')] # Tự định nghĩa mốc thủ công

    labels = range(1, len(bins))
    
    # Gán sao (rating) dựa trên số lần nghe rơi vào khoảng nào
    df['rating'] = pd.cut(df['playcount'], bins=bins, labels=labels, include_lowest=True).astype(int)

    print("\n3. Ánh xạ ID thành Index liên tục (Label Encoding) cho Neural Network...")
    df['user_index'] = pd.factorize(df['userID'])[0]
    df['song_index'] = pd.factorize(df['itemID'])[0]
    
    # Lấy thông số cấu hình mạng nơ-ron
    num_users = df['user_index'].nunique()
    num_songs = df['song_index'].nunique()
    print(f"-> Tổng số User: {num_users} | Tổng số Bài hát: {num_songs}")

    # Xuất ra Dataframe chuẩn cuối cùng
    ncf_input_df = df[['user_index', 'song_index', 'rating']]
    
    print("\n[THÀNH CÔNG] Dữ liệu Input đã sẵn sàng đưa vào mạng Keras NCF:")
    print(ncf_input_df)
    
    return ncf_input_df, num_users, num_songs

# Chạy hàm xử lý
ncf_data, NUM_USERS, NUM_SONGS = prepare_lastfm_data()

1. Đang nạp dữ liệu Last.fm (Dữ liệu thô)...
Dữ liệu thô ban đầu:
  userID itemID  playcount
0  U_001    T_A        500
1  U_001    T_B         15
2  U_001    T_C          2
3  U_002    T_A        350
4  U_002    T_C         60

2. Quy đổi Playcount thành Rating (1 -> 5 sao) bằng Quantile...

3. Ánh xạ ID thành Index liên tục (Label Encoding) cho Neural Network...
-> Tổng số User: 4 | Tổng số Bài hát: 4

[THÀNH CÔNG] Dữ liệu Input đã sẵn sàng đưa vào mạng Keras NCF:
   user_index  song_index  rating
0           0           0       5
1           0           1       2
2           0           2       1
3           1           0       5
4           1           2       3
5           2           1       1
6           2           3       4
7           3           0       3
